# Amazon Bedrock AgentCore Runtime에서 Amazon Bedrock 모델 기반 LangGraph agent 호스팅

## 개요

이 튜토리얼에서는 Amazon Bedrock AgentCore Runtime을 사용하여 기존 에이전트를 호스팅하는 방법을 알아봅니다. 

Amazon Bedrock 모델을 사용하는 LangGraph 예제를 중점적으로 살펴봅니다. Amazon Bedrock 모델 기반 Strands Agents 예제는 [여기](../01-strands-with-bedrock-model),
OpenAI 모델 기반 Strands Agents 예제는 [여기](../03-strands-with-openai-model)에서 확인할 수 있습니다.

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                    |
|:--------------------|:-----------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                       |
| 에이전트 유형       | 단일                                                                         |
| Agentic Framework   | LangGraph                                                                    |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                  |
| 튜토리얼 구성 요소  | AgentCore Runtime에 에이전트 호스팅, LangGraph 및 Amazon Bedrock 모델 사용  |
| 튜토리얼 분야       | 산업 공통                                                                    |
| 예제 난이도         | 쉬움                                                                         |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 boto3                                 |

### 튜토리얼 아키텍처

이 튜토리얼에서는 기존 에이전트를 AgentCore Runtime에 배포하는 방법을 설명합니다. 

데모에서는 Amazon Bedrock 모델을 사용하는 LangGraph agent를 사용합니다.

예제에서는 `get_weather`와 `get_time` 두 가지 tool이 포함된 매우 간단한 에이전트를 사용합니다. 

<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="50%"/>
</div>

### 튜토리얼 주요 기능

* Amazon Bedrock AgentCore Runtime에 에이전트 호스팅
* Amazon Bedrock 모델 사용
* LangGraph 사용


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* LangGraph
* Docker running

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## 에이전트 생성 및 로컬 실험

에이전트를 AgentCore Runtime에 배포하기 전에 로컬에서 개발하고 실행하여 실험합니다.

production agentic 애플리케이션에서는 에이전트 생성 과정과 호출 과정을 분리해야 합니다. AgentCore Runtime에서는 에이전트 호출 부분에 `@app.entrypoint` decorator를 적용하여 Runtime의 entry point로 사용합니다. 먼저 실험 단계에서 에이전트를 개발하는 방법을 살펴봅니다.

이 단계의 아키텍처는 다음과 같습니다.

<div style="text-align:left">
    <img src="images/architecture_local.png" width="60%"/>
</div>

In [ ]:
%%writefile langgraph_bedrock.py
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
import argparse
import json
import operator
import math

# calculator tool 생성
@tool
def calculator(expression: str) -> str:
    """
    Calculate the result of a mathematical expression.
    
    Args:
        expression: A mathematical expression as a string (e.g., "2 + 3 * 4", "sqrt(16)", "sin(pi/2)")
    
    Returns:
        The result of the calculation as a string
    """
    try:
        # expression에서 사용할 수 있는 안전한 함수 정의
        safe_dict = {
            "__builtins__": {},
            "abs": abs, "round": round, "min": min, "max": max,
            "sum": sum, "pow": pow,
            # 수학 함수
            "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos, "tan": math.tan,
            "log": math.log, "log10": math.log10, "exp": math.exp,
            "pi": math.pi, "e": math.e,
            "ceil": math.ceil, "floor": math.floor,
            "degrees": math.degrees, "radians": math.radians,
            # 기본 operator(명시적 사용)
            "add": operator.add, "sub": operator.sub,
            "mul": operator.mul, "truediv": operator.truediv,
        }
        
        # expression을 안전하게 평가
        result = eval(expression, safe_dict)
        return str(result)
        
    except ZeroDivisionError:
        return "Error: Division by zero"
    except ValueError as e:
        return f"Error: Invalid value - {str(e)}"
    except SyntaxError:
        return "Error: Invalid mathematical expression"
    except Exception as e:
        return f"Error: {str(e)}"

# custom weather tool 생성
@tool
def weather():
    """Get weather"""  # Dummy 구현
    return "sunny"

# 수동 LangGraph 구성으로 에이전트 정의
def create_agent():
    """LangGraph 에이전트를 생성하고 구성합니다."""
    from langchain_aws import ChatBedrock
    
    # LLM 초기화(필요에 따라 모델과 parameter 조정)
    llm = ChatBedrock(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # 또는 선호하는 모델
        model_kwargs={"temperature": 0.1}
    )
    
    # tool을 LLM에 binding
    tools = [calculator, weather]
    llm_with_tools = llm.bind_tools(tools)
    
    # system message 설정
    system_message = "You're a helpful assistant. You can do simple math calculation, and tell the weather."
    
    # chatbot node 정의
    def chatbot(state: MessagesState):
        # 아직 없으면 system message 추가
        messages = state["messages"]
        if not messages or not isinstance(messages[0], SystemMessage):
            messages = [SystemMessage(content=system_message)] + messages
        
        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}
    
    # graph 생성
    graph_builder = StateGraph(MessagesState)
    
    # node 추가
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))
    
    # edge 추가
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")
    
    # entry point 설정
    graph_builder.set_entry_point("chatbot")
    
    # graph compile 수행
    return graph_builder.compile()

# 에이전트 초기화
agent = create_agent()

def langgraph_bedrock(payload):
    """
    payload로 에이전트를 호출합니다.
    """
    user_input = payload.get("prompt")
    
    # LangGraph에서 예상하는 형식으로 input 생성
    response = agent.invoke({"messages": [HumanMessage(content=user_input)]})
    
    # 최종 message content 추출
    return response["messages"][-1].content

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = langgraph_bedrock(json.loads(args.payload))
    print(response)

#### 로컬 에이전트 호출

In [ ]:
!python langgraph_bedrock.py '{"prompt": "What is the weather now?"}'

## AgentCore Runtime 배포를 위한 에이전트 준비

이제 에이전트를 AgentCore Runtime에 배포합니다. 이를 위해 다음 작업이 필요합니다.
* `from bedrock_agentcore.runtime import BedrockAgentCoreApp`으로 Runtime App 가져오기
* 코드에서 `app = BedrockAgentCoreApp()`으로 App 초기화
* 호출 함수에 `@app.entrypoint` decorator 적용
* `app.run()`으로 AgentCoreRuntime이 에이전트 실행을 제어하도록 설정

### Amazon Bedrock 모델 기반 LangGraph
Amazon Bedrock 모델을 사용하는 LangGraph부터 시작합니다. 다른 framework와 모델을 사용하는 예제는 상위 디렉터리에서 확인할 수 있습니다.

In [ ]:
%%writefile langgraph_bedrock.py
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import argparse
import json
import operator
import math

app = BedrockAgentCoreApp()

# calculator tool 생성
@tool
def calculator(expression: str) -> str:
    """
    Calculate the result of a mathematical expression.
    
    Args:
        expression: A mathematical expression as a string (e.g., "2 + 3 * 4", "sqrt(16)", "sin(pi/2)")
    
    Returns:
        The result of the calculation as a string
    """
    try:
        # expression에서 사용할 수 있는 안전한 함수 정의
        safe_dict = {
            "__builtins__": {},
            "abs": abs, "round": round, "min": min, "max": max,
            "sum": sum, "pow": pow,
            # 수학 함수
            "sqrt": math.sqrt, "sin": math.sin, "cos": math.cos, "tan": math.tan,
            "log": math.log, "log10": math.log10, "exp": math.exp,
            "pi": math.pi, "e": math.e,
            "ceil": math.ceil, "floor": math.floor,
            "degrees": math.degrees, "radians": math.radians,
            # 기본 operator(명시적 사용)
            "add": operator.add, "sub": operator.sub,
            "mul": operator.mul, "truediv": operator.truediv,
        }
        
        # expression을 안전하게 평가
        result = eval(expression, safe_dict)
        return str(result)
        
    except ZeroDivisionError:
        return "Error: Division by zero"
    except ValueError as e:
        return f"Error: Invalid value - {str(e)}"
    except SyntaxError:
        return "Error: Invalid mathematical expression"
    except Exception as e:
        return f"Error: {str(e)}"

# custom weather tool 생성
@tool
def weather():
    """Get weather"""  # Dummy 구현
    return "sunny"

# 수동 LangGraph 구성으로 에이전트 정의
def create_agent():
    """LangGraph 에이전트를 생성하고 구성합니다."""
    from langchain_aws import ChatBedrock
    
    # LLM 초기화(필요에 따라 모델과 parameter 조정)
    llm = ChatBedrock(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # 또는 선호하는 모델
        model_kwargs={"temperature": 0.1}
    )
    
    # tool을 LLM에 binding
    tools = [calculator, weather]
    llm_with_tools = llm.bind_tools(tools)
    
    # system message 설정
    system_message = "You're a helpful assistant. You can do simple math calculation, and tell the weather."
    
    # chatbot node 정의
    def chatbot(state: MessagesState):
        # 아직 없으면 system message 추가
        messages = state["messages"]
        if not messages or not isinstance(messages[0], SystemMessage):
            messages = [SystemMessage(content=system_message)] + messages
        
        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}
    
    # graph 생성
    graph_builder = StateGraph(MessagesState)
    
    # node 추가
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))
    
    # edge 추가
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")
    
    # entry point 설정
    graph_builder.set_entry_point("chatbot")
    
    # graph compile 수행
    return graph_builder.compile()

# 에이전트 초기화
agent = create_agent()

@app.entrypoint
def langgraph_bedrock(payload):
    """
    payload로 에이전트를 호출합니다.
    """
    user_input = payload.get("prompt")
    
    # LangGraph에서 예상하는 형식으로 input 생성
    response = agent.invoke({"messages": [HumanMessage(content=user_input)]})
    
    # 최종 message content 추출
    return response["messages"][-1].content

if __name__ == "__main__":
    app.run()

## 내부 동작

`BedrockAgentCoreApp`을 사용하면 다음 작업이 자동으로 수행됩니다.

* port 8080에서 수신 대기하는 HTTP server 생성
* 에이전트 요청 처리를 위한 필수 `/invocations` endpoint 구현
* health check를 위한 `/ping` endpoint 구현(asynchronous agent에 매우 중요)
* 적절한 content type 및 응답 형식 처리
* AWS 표준에 따른 오류 처리 관리

## AgentCore Runtime에 에이전트 배포

`CreateAgentRuntime` operation은 container image, 환경 변수, 암호화 설정을 지정할 수 있는 포괄적인 구성 옵션을 지원합니다. protocol 설정(HTTP, MCP)과 권한 부여 메커니즘을 구성하여 client가 에이전트와 통신하는 방식도 제어할 수 있습니다. 

**참고:** 운영 환경에서는 코드를 container로 package하고 CI/CD pipeline과 IaC를 사용하여 ECR에 push하는 것이 좋습니다.

이 튜토리얼에서는 Amazon Bedrock AgentCore Python SDK를 사용하여 artifact를 간편하게 package하고 AgentCore Runtime에 배포합니다.

### AgentCore Runtime 배포 구성

먼저 starter toolkit을 사용하여 entrypoint, 앞에서 생성한 execution role, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 실행 시 Amazon ECR repository를 자동으로 생성하도록 starter toolkit도 구성합니다.

configure 단계에서는 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

<div style="text-align:left">
    <img src="images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()

agent_name = "langgraph_claude_getting_started"
response = agentcore_runtime.configure(
    entrypoint="langgraph_bedrock.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
)
response

### AgentCore Runtime에 에이전트 실행

Dockerfile이 준비되었으므로 AgentCore Runtime에 에이전트를 실행합니다. 이 과정에서 Amazon ECR repository와 AgentCore Runtime이 생성됩니다.

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch()

### AgentCore Runtime 상태 확인
AgentCore Runtime을 배포했으므로 배포 상태를 확인합니다.

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

### AgentCore Runtime 호출

이제 payload로 AgentCore Runtime을 호출할 수 있습니다.

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "How much is 2+2?"})
invoke_response

### 호출 결과 처리

이제 호출 결과를 처리하여 애플리케이션에 포함할 수 있습니다.

In [ ]:
from IPython.display import Markdown, display
import json

response_text = invoke_response["response"][0]
display(Markdown(response_text))

### boto3로 AgentCore Runtime 호출

AgentCore Runtime이 생성되었으므로 어떤 AWS SDK로도 호출할 수 있습니다. 예를 들어 boto3의 `invoke_agent_runtime` method를 사용할 수 있습니다.

In [ ]:
import boto3

agent_arn = launch_result.agent_arn
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 2+2?"}),
)

# lifecycle 관리를 위해 runtime session ID 저장
runtime_session_id = boto3_response.get("runtimeSessionId")
print(f"Runtime Session ID: {runtime_session_id}")

if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

### Session 중지

개별 session이 더 이상 필요하지 않으면 중지해야 합니다.
이렇게 하면 Runtime은 새 session을 위해 계속 실행하면서 해당 session의 microVM 리소스를 해제합니다.
아래에서 `stop_runtime_session`을 살펴봅니다.

In [ ]:
# --- Inline Session Lifecycle 데모 ---
# stop_runtime_session은 Runtime을 새 session용으로 유지하면서 이 session의 microVM 리소스를 해제함


if runtime_session_id:
    agentcore_client.stop_runtime_session(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=runtime_session_id,
        qualifier="DEFAULT",
    )
    print(f"✅ Session '{runtime_session_id}' stopped — microVM resources released")
else:
    print("⚠️ No session ID available to stop")

### Lifecycle 구성 데모

이제 더 짧은 idle timeout으로 Runtime을 구성하는 방법을 살펴봅니다.
5분(300초) idle timeout을 사용하는 두 번째 Runtime을 생성하여 lifecycle 구성이
session 동작에 미치는 영향을 확인합니다. 두 Runtime은 함께 존재합니다.

In [ ]:
# --- Lifecycle 구성 데모 ---
# production에서는 workload에 적합한 timeout 선택:
#   - 개발/테스트: 5~15분
#   - 대화형 session: 30~60분
#   - 장기 실행 workload: 필요에 따라 조정
#

agentcore_runtime_short = Runtime()
agent_name_short = "langgraph_claude_short_timeout"

# 더 짧은 idle timeout으로 구성
response_short = agentcore_runtime_short.configure(
    entrypoint="langgraph_bedrock.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name_short,
)

# 두 번째 Runtime 실행
launch_result_short = agentcore_runtime_short.launch()
print(f"Second runtime launched: {launch_result_short.agent_id}")

# 준비될 때까지 대기
status_response_short = agentcore_runtime_short.status()
status_short = status_response_short.endpoint["status"]
while status_short not in ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]:
    time.sleep(10)
    status_response_short = agentcore_runtime_short.status()
    status_short = status_response_short.endpoint["status"]
    print(f"Short timeout runtime status: {status_short}")

# 이제 boto3를 사용하여 더 짧은 idle timeout으로 Runtime 업데이트
# UpdateAgentRuntime은 전체 교체 API이므로 모든 필수 field를 다시 제공해야 함
# 먼저 현재 Runtime 구성 가져오기
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
current_runtime = agentcore_control_client.get_agent_runtime(agentRuntimeId=launch_result_short.agent_id)

update_response = agentcore_control_client.update_agent_runtime(
    agentRuntimeId=launch_result_short.agent_id,
    agentRuntimeArtifact=current_runtime["agentRuntimeArtifact"],
    roleArn=current_runtime["roleArn"],
    networkConfiguration=current_runtime["networkConfiguration"],
    lifecycleConfiguration={
        "idleRuntimeSessionTimeout": 300  # 5분
    },
)
print("✅ Runtime updated with 5-minute idle timeout")

# 두 번째 Runtime을 호출하여 작동 확인
invoke_response_short = agentcore_runtime_short.invoke({"prompt": "What is 3+3?"})
print(f"Second runtime response: {invoke_response_short['response'][0]}")

## 리소스 정리

이제 AgentCore Runtime과 관련 리소스를 정리합니다. 불필요한 비용을 방지하도록 Runtime을 먼저 삭제한 뒤 ECR repository 같은 지원 리소스를 정리합니다.

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split("/")[1]

In [ ]:
# --- 활성 session을 중지하여 microVM 리소스 해제 ---
import boto3

agentcore_client = boto3.client("bedrock-agentcore", region_name=region)
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

# 활성 session을 중지하여 해당 microVM 리소스 해제
# production에서는 이 방식으로 Runtime을 유지하면서 개별 user session 종료
# AgentCore Runtime 비용은 vCPU와 Memory를 기준으로 하므로 session 중지로 불필요한 비용 방지
# 참고: 이전 데모 셀에서 session이 이미 중지되었다면 다음 오류가 발생함
# ResourceNotFoundException은 except block에서 적절히 처리함
if "runtime_session_id" in locals() and runtime_session_id:
    try:
        agentcore_client.stop_runtime_session(
            agentRuntimeArn=launch_result.agent_arn,
            runtimeSessionId=runtime_session_id,
            qualifier="DEFAULT",
        )
        print(f"✅ Session '{runtime_session_id}' stopped")
    except Exception as e:
        print(f"⚠️ Failed to stop session '{runtime_session_id}': {e}")

# --- 두 Runtime 모두 삭제 ---
# 원본 Runtime
try:
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )
    print(f"✅ Original runtime '{launch_result.agent_id}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete original runtime: {e}")

# 짧은 timeout Runtime
if "launch_result_short" in locals():
    try:
        agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=launch_result_short.agent_id,
        )
        print(f"✅ Short-timeout runtime '{launch_result_short.agent_id}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete short-timeout runtime: {e}")

# --- ECR repository 삭제 ---
try:
    ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)
    print(f"✅ ECR repository '{launch_result.ecr_uri.split('/')[1]}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete ECR repository: {e}")

if "launch_result_short" in locals():
    try:
        ecr_client.delete_repository(repositoryName=launch_result_short.ecr_uri.split("/")[1], force=True)
        print(f"✅ Second ECR repository '{launch_result_short.ecr_uri.split('/')[1]}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete second ECR repository: {e}")

# 축하합니다!